In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "Qwen/Qwen3-4B"
tok = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id, torch_dtype=torch.float16, device_map="auto"
)

def ask(question, max_new_tokens=300):
    messages = [{"role": "user", "content": question}]
    text = tok.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
        enable_thinking=False
    )
    inputs = tok(text, return_tensors="pt").to(model.device)
    output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return tok.decode(output[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

print(ask("What is the capital of Ethiopia?"))

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

The capital of Ethiopia is **Addis Ababa**. It is the political, economic, and cultural center of the country and has been the capital since 1948.


In [2]:
import pandas as pd

df = pd.read_csv("amharic_proverbs.csv")

def from_amharic(row):
    return f"What does this Amharic proverb mean? Answer in 2-3 sentences. {row['proverb_amharic']}"

def from_english(row):
    return ("This is an Ethiopian proverb translated word for word into English: "
            f"\"{row['proverb_english']}\". What does it mean? Answer in 2-3 sentences.")

df["answer_from_amharic"] = [ask(from_amharic(r)) for _, r in df.iterrows()]
df["answer_from_english"] = [ask(from_english(r)) for _, r in df.iterrows()]

df.to_csv("results.csv", index=False, encoding="utf-8-sig")
df[["proverb_amharic", "true_meaning", "answer_from_amharic", "answer_from_english"]]

,proverb_amharic,true_meaning,answer_from_amharic,answer_from_english
0,የፈሰሰ ውሃ አይታፈስም,What is done cannot be undone,"The Amharic proverb ""የፈሰሰ ውሃ አይታፈስም"" translate...","The Ethiopian proverb ""Spilled water cannot be..."
1,ቀስ በቀስ እንቁላል በእግሩ ይሄዳል,"With patience, things develop in their own time","The Amharic proverb ""ቀስ በቀስ እንቁላል በእግሩ ይሄዳል"" m...","The Ethiopian proverb ""Slowly, slowly, the egg..."
2,ውሻ በቀደደው ጅብ ይገባል,A small weakness opens the way for a much bigg...,"The Amharic proverb ""ውሻ በቀደደው ጅብ ይገባል"" transla...","The Ethiopian proverb ""Through the hole the do..."
3,ጅብ ከሄደ ውሻ ጮኸ,Acting after the danger has passed is useless,"The Amharic proverb ""ጅብ ከሄደ ውሻ ጮኸ"" translates ...","The Ethiopian proverb ""The dog barked after th..."
4,ከድጡ ወደ ማጡ,Escaping one problem only to fall into a worse...,"The Amharic proverb ""ከድጡ ወደ ማጡ"" means ""from a ...","The Ethiopian proverb ""From the mud into the s..."
5,ያልጠረጠረ ተመነጠረ,"Be cautious, because the unsuspecting get hurt","The Amharic proverb ""ያልጠረጠረ ተመነጠረ"" translates ...","The Ethiopian proverb ""The one who did not sus..."
6,የጠመጠመ ሁሉ ቄስ አይደለም,Appearances can be deceiving,"The Amharic proverb ""የጠመጠመ ሁሉ ቄስ አይደለም"" transl...","The Ethiopian proverb ""Not everyone who wears ..."
7,አፉ ቅቤ ልቡ ጩቤ,Sweet words can hide harmful intentions,"The Amharic proverb ""አፉ ቅቤ ልቡ ጩቤ"" translates t...","The Ethiopian proverb ""His mouth is butter, hi..."
8,የቸኮለ አፍስሶ ለቀመ,Rushing causes mistakes and extra work,"The Amharic proverb ""የቸኮለ አፍስሶ ለቀመ"" translates...","The Ethiopian proverb ""The one who hurried spi..."
9,ጥቂት በጥቂት ይሞላል ልቃቂት,Small steady efforts add up over time,"The Amharic proverb ""ጥቂት በጥቂት ይሞላል ልቃቂት"" means...","The Ethiopian proverb ""Little by little the ba..."


In [3]:
# from google.colab import files
# files.download("results.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [4]:
for text in ["ድር ቢያብር አንበሳ ያስር", "When spider webs unite, they can tie up a lion"]:
    print(len(tok(text).input_ids), "tokens:", text)

19 tokens: ድር ቢያብር አንበሳ ያስር
11 tokens: When spider webs unite, they can tie up a lion
